<a href="https://colab.research.google.com/github/alj-x64/Realtime-Bass-Tablature-Transcription/blob/main/Adaptive_Multi_Stage_HPO.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import random
import math
import copy

class AdaptiveMultiStageHPO:
    def __init__(self, total_budget=30, alpha=0.5, beta=0.5, rho_s=0.60, rho_r=0.40, gamma=0.30):
        """
        Inisyalisasyon ng optimizer base sa mathematical constraints ng thesis mo.
        """
        self.B = total_budget
        self.alpha = alpha
        self.beta = beta

        # Thesis Allocation Ratios (Table 3)
        self.rho_S = rho_s
        self.rho_R = rho_r
        self.gamma = gamma

        # Kalkulahin ang Budgets (Equations 9 - 18)
        self.S = int(self.rho_S * self.B)
        self.R = int(self.rho_R * self.B)

        # Eq 16: P = ceil(gamma * rho_S * B)
        self.P = max(2, math.ceil(self.gamma * self.S)) # Siguraduhing may at least 2 para makapag-crossover

        # Eq 17: G = floor(rho_S * B / P)
        self.G = max(1, math.floor(self.S / self.P))

        print(f"--- HPO Budgets Initialized ---")
        print(f"Total Budget (B): {self.B} trials")
        print(f"Selection Budget (S): {self.S} | Refinement Budget (R): {self.R}")
        print(f"Population Size (P): {self.P} | Generations (G): {self.G}")
        print(f"-------------------------------")

        # Search Space (Table 2)
        self.search_space = {
            'learning_rate': {'type': 'continuous', 'range': [1e-5, 1e-2]},
            'dropout_rate': {'type': 'continuous', 'range': [0.1, 0.5]},
            'activation': {'type': 'categorical', 'values': ['ReLU', 'Tanh', 'ELU']},
            'conv_layers': {'type': 'discrete', 'range': [1, 4]},
            'filter_layers': {'type': 'categorical', 'values': [16, 32, 64, 128]},
            'kernel_size': {'type': 'categorical', 'values': [2, 3, 5, 7]}
        }

    def generate_individual(self):
        """Gumagawa ng random na hyperparameter configuration vector (Eq 7)."""
        ind = {}
        for key, params in self.search_space.items():
            if params['type'] == 'continuous':
                # Log-uniform para sa learning rate, uniform para sa dropout
                if key == 'learning_rate':
                    ind[key] = 10 ** np.random.uniform(np.log10(params['range'][0]), np.log10(params['range'][1]))
                else:
                    ind[key] = np.random.uniform(params['range'][0], params['range'][1])
            elif params['type'] == 'discrete':
                # Nilagyan ng + 1 para maging inclusive ang maximum limit sa randint
                ind[key] = np.random.randint(params['range'][0], params['range'][1] + 1)
            elif params['type'] == 'categorical':
                ind[key] = random.choice(params['values'])
        return ind

    def fitness_function(self, loss, latency):
        """Kino-compute ang fitness kung saan balance ang accuracy at latency (Eq 19 at Eq 39)."""
        latency_penalty = (latency / 200.0) if latency > 0 else 0
        return self.alpha * (1.0 - loss) - self.beta * latency_penalty

    def crossover(self, parent1, parent2):
        """Uniform crossover na may 0.5 probability (Eq 30)."""
        offspring = {}
        for key in self.search_space.keys():
            offspring[key] = parent1[key] if random.random() < 0.5 else parent2[key]
        return offspring

    def mutate(self, individual):
        """Nag-a-apply ng task-dependent mutation na may Pm = 0.5 (Eq 31 - 33)."""
        mutated = copy.deepcopy(individual)
        for key, params in self.search_space.items():
            if random.random() < 0.5: # 50% mutation chance kada gene
                if params['type'] == 'continuous':
                    # Gaussian mutation (Eq 33)
                    sigma = (params['range'][1] - params['range'][0]) * 0.1 # 10% ng range width
                    new_val = mutated[key] + np.random.normal(0, sigma)
                    mutated[key] = np.clip(new_val, params['range'][0], params['range'][1]) # Iwas out-of-bounds
                elif params['type'] == 'discrete':
                    # Random resetting
                    mutated[key] = np.random.randint(params['range'][0], params['range'][1] + 1)
                elif params['type'] == 'categorical':
                    # Random resetting
                    mutated[key] = random.choice(params['values'])
        return mutated

    def run_optimization(self, train_eval_fn):
        """
        Pinapatakbo ang buong Selection at Refinement stages.
        Ang `train_eval_fn` ay ang callback sa main.py na humahawak sa PyTorch training.
        """
        # --- SELECTION STAGE ---
        population = [{'config': self.generate_individual(), 'fitness': 0, 'loss': 0, 'latency': 0} for _ in range(self.P)]

        all_losses = [] # Para i-track lahat ng losses (kailangan sa P25 calculation sa huli)

        for gen in range(self.G):
            print(f"\n--- Generation {gen + 1}/{self.G} ---")

            # I-evaluate ang bawat model sa Population
            for ind in population:
                # Dito tinatawag yung function galing sa main.py
                loss, latency = train_eval_fn(ind['config'], stress_test=False)
                ind['loss'] = loss
                ind['latency'] = latency
                ind['fitness'] = self.fitness_function(loss, latency)
                all_losses.append(loss)

            # Rank at Select (Eq 28 - 29)
            population.sort(key=lambda x: x['fitness'], reverse=True)
            survivors = population[:max(1, len(population)//2)]

            # Breed ng Next Generation
            next_generation = copy.deepcopy(survivors)
            while len(next_generation) < self.P:
                # Kumuha ng dalawang random parents galing sa mga survivors
                p1, p2 = random.sample(survivors, 2) if len(survivors) > 1 else (survivors[0], survivors[0])
                offspring_config = self.crossover(p1['config'], p2['config'])
                mutated_config = self.mutate(offspring_config)
                next_generation.append({'config': mutated_config, 'fitness': 0, 'loss': 0, 'latency': 0})

            population = next_generation

        # --- REFINEMENT STAGE ---
        print("\n--- Entering Refinement Stage ---")
        # I-evaluate ulit ang final population para sa final ranking
        for ind in population:
            if ind['fitness'] == 0: # I-evaluate lang yung mga bagong anak na wala pang score
                loss, latency = train_eval_fn(ind['config'], stress_test=False)
                ind['loss'], ind['latency'] = loss, latency
                ind['fitness'] = self.fitness_function(loss, latency)
                all_losses.append(loss)

        population.sort(key=lambda x: x['fitness'], reverse=True)
        p25_loss = np.percentile(all_losses, 25) # Kunin ang 25th percentile ng lahat ng naitalang loss

        for rank, candidate in enumerate(population):
            if rank >= self.R:
                break # Tumigil na kung ubos na ang refinement budget (R)

            print(f"Testing Candidate {rank + 1}...")

            # 1. HPO TRIGGER: I-uutos ng algorithm na lagyan ng Gaussian Noise ang tensors sa PyTorch
            stress_loss, stress_latency = train_eval_fn(candidate['config'], stress_test=True)

            # 2. HPO PENALTY (LATENCY JITTER): Part ng Algorithm mo (Simulate system interrupts)
            if np.random.rand() < 0.10: # 10% trigger probability ng OS lag
                jitter = np.random.uniform(0, 50)
                stress_latency += jitter
                print(f"   [HPO Algorithm] Applied {jitter:.2f}ms Latency Jitter penalty!")

            print(f"Candidate {rank + 1} Results -> Loss: {stress_loss:.4f} (Req <= {p25_loss:.4f}), Latency: {stress_latency:.2f}ms (Req <= 200ms)")

            # Constraint-aware check (Eq 40)
            if stress_latency <= 200.0 and stress_loss <= p25_loss:
                print(f"\n>>> OPTIMAL CONFIGURATION FOUND (Candidate {rank + 1}) <<<")
                return candidate['config']
            else:
                print("Candidate failed strict constraints. Backtracking to next best...")

        print("\n>>> NO CANDIDATE PASSED ALL STRICT CONSTRAINTS. RETURNING BEST BEST-EFFORT CONFIG <<<")
        return population[0]['config']